# Visual Product Search — Kaggle Backend Server
**Exposes two endpoints via ngrok:**
- `POST /crop` — YOLO crop of uploaded image
- `POST /search` — CLIP embed + FAISS retrieval + BLIP-2 ITM re-rank

**Run all cells top to bottom. Copy the ngrok URL printed at the end into your Streamlit app.**

## 0. Install Dependencies

In [4]:
%%capture
!pip install -q flask flask-cors pyngrok
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q faiss-cpu
!pip install -q transformers>=4.36 accelerate sentencepiece protobuf
!pip install -q Pillow opencv-python-headless
%pip install -q ultralytics

## 1. Configuration — Edit Paths Here

In [5]:
import os
from pathlib import Path

# ── Artifact paths (adjust to your Kaggle dataset slugs) ─────────────────────
WORK_DIR         = Path('/kaggle/working')
RESULTS_DIR      = WORK_DIR / 'results_C'

# Gallery index & IDs from Config C (alpha=0.7)
ALPHA            = 0.7
SEED             = 2023085   # seed whose checkpoint to use (change if needed)

#GALLERY_INDEX_PATH = str(WORK_DIR / f'gallery_index_C_seed{SEED}_a{ALPHA}.faiss')
GALLERY_IDS_PATH   = '/kaggle/input/datasets/venkat96r/gal-emb/gal_embs_C_0.7.npy'   # item_id per gallery row
CAPTION_CACHE_PATH = '/kaggle/input/datasets/venkat96r/caption2/caption_cache.json'  # or caption_cache.json if reused from B

# Dataset image root — needed to serve gallery image paths back to client
DATASET_BASE     = Path('/kaggle/input/datasets/venkat96r/vr-inshop-dataset/img')
IMG_ROOT         = DATASET_BASE
PARTITION_FILE   = DATASET_BASE / 'list_eval_partition.txt'

# Model weights
YOLO_WEIGHTS     = '/kaggle/input/datasets/venkat96r/vr-final-yolo/best_yolo_fine_tuned_without_augmenatation.pt'
CLIP_CKPT_PATH   = '/kaggle/input/datasets/venkatrrr/clip-finetuned-085/best_clip_C_finetuned.pt'

# Model config
CLIP_MODEL       = 'ViT-B/32'
BLIP2_MODEL      = 'Salesforce/blip2-opt-2.7b'
UNFREEZE_BLOCKS  = 4
TOP_K_DEFAULT    = 10
ITM_TOP_N        = 30        # retrieve this many from FAISS, then re-rank to TOP_K_DEFAULT
DTYPE            = 'bfloat16'

# ngrok auth token — get yours free at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = '33CVblnubvJ0XkO6OFIoahA1ayu_9wXpjRCRthtwcvgUPQVq'   # ← PASTE YOUR TOKEN

print('Config loaded.')
print(f'Gallery index : {GALLERY_IDS_PATH}')
print(f'CLIP checkpoint: {CLIP_CKPT_PATH}')

Config loaded.
Gallery index : /kaggle/input/datasets/venkat96r/gal-emb/gal_embs_C_0.7.npy
CLIP checkpoint: /kaggle/input/datasets/venkatrrr/clip-finetuned-085/best_clip_C_finetuned.pt


## 2. Imports

In [6]:
import json, gc, base64, io, warnings, copy
import numpy as np
import torch
import clip
import faiss
from PIL import Image
from pathlib import Path
from ultralytics import YOLO
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import torch.nn as nn

warnings.filterwarnings('ignore')
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_GPUS = torch.cuda.device_count()
TORCH_DTYPE = torch.bfloat16
print(f'Device: {DEVICE} | GPUs: {NUM_GPUS}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Device: cuda | GPUs: 2


## 3. Parse Gallery Metadata (path lookup)

In [7]:
def parse_partition(partition_file, img_root):
    train, query, gallery = [], [], []
    with open(partition_file, 'r') as f:
        lines = f.read().splitlines()
    for line in lines[2:]:
        parts = line.split()
        if len(parts) < 3:
            continue
        rel_path, item_id, split = parts[0], parts[1], parts[2].lower().strip()
        abs_path = str(Path(img_root) / rel_path)
        entry = (abs_path, item_id)
        if split == 'train':   train.append(entry)
        elif split == 'query': query.append(entry)
        elif split == 'gallery': gallery.append(entry)
    return train, query, gallery

print('Parsing dataset partition...')
train_data, query_data, gallery_data = parse_partition(PARTITION_FILE, IMG_ROOT)
print(f'Gallery: {len(gallery_data):,} | Query: {len(query_data):,} | Train: {len(train_data):,}')

# Index: gallery row → (path, item_id)
gallery_row_to_meta = {i: (path, iid) for i, (path, iid) in enumerate(gallery_data)}

Parsing dataset partition...
Gallery: 12,612 | Query: 14,218 | Train: 25,882


## 4. Load YOLO

In [8]:
yolo = YOLO(YOLO_WEIGHTS)
yolo.to(DEVICE)
print('YOLO loaded ✓')

def yolo_crop(pil_img: Image.Image, conf_thresh: float = 0.25) -> Image.Image:
    """Run YOLO on a PIL image, return best-confidence crop. Falls back to full image."""
    img_rgb = pil_img.convert('RGB')
    results  = yolo(img_rgb, conf=conf_thresh, verbose=False)
    boxes    = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return img_rgb
    best_idx        = boxes.conf.argmax().item()
    x1, y1, x2, y2 = boxes.xyxy[best_idx].cpu().numpy().astype(int)
    x1, y1          = max(0, x1), max(0, y1)
    crop = img_rgb.crop((x1, y1, x2, y2))
    return crop if crop.width > 10 and crop.height > 10 else img_rgb

YOLO loaded ✓


## 5. Load Fine-tuned CLIP

In [10]:
def build_clip_for_finetuning(model_name, device, unfreeze_blocks):
    model, preprocess = clip.load(model_name, device=device)
    model = model.float()
    for p in model.parameters():
        p.requires_grad_(False)
    vis_blocks = model.visual.transformer.resblocks
    n_total    = len(vis_blocks)
    for blk in vis_blocks[n_total - unfreeze_blocks:]:
        for p in blk.parameters():
            p.requires_grad_(True)
    if hasattr(model.visual, 'ln_post'):
        for p in model.visual.ln_post.parameters():
            p.requires_grad_(True)
    if hasattr(model.visual, 'proj') and model.visual.proj is not None:
        model.visual.proj.requires_grad_(True)
    return model, preprocess

print('Loading fine-tuned CLIP...')
clip_model, clip_preprocess = build_clip_for_finetuning(CLIP_MODEL, DEVICE, UNFREEZE_BLOCKS)

ckpt = torch.load(CLIP_CKPT_PATH, map_location=DEVICE,weights_only=False)
clip_model.load_state_dict(ckpt['model_state'])
clip_model.eval()
clip_model = clip_model.to(DEVICE)

print(f'CLIP loaded from checkpoint (epoch {ckpt["epoch"]}, Recall@10={ckpt["recall10"]:.4f}) ✓')

Loading fine-tuned CLIP...
CLIP loaded from checkpoint (epoch 1, Recall@10=0.5793) ✓


## 6. Load FAISS Gallery Index

In [12]:
import numpy as np

print('Loading gallery embeddings and building FAISS index...')

# 1. Load the numpy array you already have
gal_embs = np.load('/kaggle/input/datasets/venkat96r/gal-emb/gal_embs_C_0.7.npy')

# 2. Initialize an empty FAISS HNSW index
dim = gal_embs.shape[1]
faiss_index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
faiss_index.hnsw.efConstruction = 200
faiss_index.hnsw.efSearch = 200

# 3. Add the embeddings to build it instantly
faiss_index.add(gal_embs)

print(f'FAISS index built | Vectors: {faiss_index.ntotal:,} ✓')

# (Keep your caption cache loading part exactly the same)
with open(CAPTION_CACHE_PATH) as f:
    caption_map = json.load(f)
print(f'Caption cache loaded | {len(caption_map):,} captions ✓')


Loading gallery embeddings and building FAISS index...
FAISS index built | Vectors: 12,612 ✓
Caption cache loaded | 26,830 captions ✓


## 7. Load BLIP-2 for ITM Re-ranking

In [13]:
print('Loading BLIP-2 for ITM re-ranking (this takes ~2-3 min)...')
blip_processor = Blip2Processor.from_pretrained(BLIP2_MODEL)
blip_model     = Blip2ForConditionalGeneration.from_pretrained(
    BLIP2_MODEL,
    torch_dtype=TORCH_DTYPE,
    device_map='auto',
)
blip_model.eval()
for p in blip_model.parameters():
    p.requires_grad_(False)
print('BLIP-2 loaded ✓')

Loading BLIP-2 for ITM re-ranking (this takes ~2-3 min)...


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

BLIP-2 loaded ✓


## 8. Inference Utilities

In [14]:
@torch.no_grad()
def embed_query(pil_img: Image.Image, alpha: float = ALPHA) -> np.ndarray:
    """
    Embed a (already-cropped) query image using fine-tuned CLIP.
    If alpha < 1.0, generates a caption with BLIP-2 and fuses text embedding.
    Returns L2-normalised float32 vector.
    """
    img_t = clip_preprocess(pil_img).unsqueeze(0).to(DEVICE)

    with torch.cuda.amp.autocast(dtype=TORCH_DTYPE):
        vis_emb = clip_model.encode_image(img_t).float()
        vis_emb = vis_emb / vis_emb.norm(dim=-1, keepdim=True).clamp(min=1e-8)

        if alpha < 1.0:
            # Generate a caption for the query image
            inputs = blip_processor(
                images=pil_img,
                text='a photo of a clothing item:',
                return_tensors='pt',
            )
            inputs = {k: v.to(blip_model.device) if hasattr(v, 'to') else v
                      for k, v in inputs.items()}
            with torch.no_grad():
                generated = blip_model.generate(**inputs, max_new_tokens=40, num_beams=3)
            caption = blip_processor.batch_decode(generated, skip_special_tokens=True)[0]
            caption = caption.replace('a photo of a clothing item:', '').strip()

            tok = clip.tokenize([caption], truncate=True).to(DEVICE)
            txt_emb = clip_model.encode_text(tok).float()
            txt_emb = txt_emb / txt_emb.norm(dim=-1, keepdim=True).clamp(min=1e-8)

            fused = alpha * vis_emb + (1.0 - alpha) * txt_emb
        else:
            fused   = vis_emb
            caption = ''

    fused = fused / fused.norm(dim=-1, keepdim=True).clamp(min=1e-8)
    return fused.cpu().numpy().astype('float32'), caption


@torch.no_grad()
def blip2_itm_scores(
    query_img: Image.Image,
    candidate_captions: list,
) -> np.ndarray:
    """
    Compute BLIP-2 image-text matching (ITM) scores.
    Uses conditional generation probability as a proxy for ITM:
    score = mean log-prob of generating the caption given the query image.
    Returns array of scores, higher = better match.
    """
    scores = []
    for cap in candidate_captions:
        if not cap:
            scores.append(0.0)
            continue
        try:
            inputs = blip_processor(
                images=query_img,
                text=cap,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=64,
            )
            inputs = {k: v.to(blip_model.device) if hasattr(v, 'to') else v
                      for k, v in inputs.items()}
            # Use teacher-forcing: pass labels = input_ids to get per-token log-probs
            labels = inputs.get('input_ids', None)
            if labels is not None:
                out = blip_model(**inputs, labels=labels.clone())
                # Negative loss = log-likelihood (higher = better match)
                scores.append(-out.loss.item())
            else:
                scores.append(0.0)
        except Exception:
            scores.append(0.0)
    return np.array(scores, dtype='float32')


def pil_to_b64(img: Image.Image, max_size: int = 300) -> str:
    """Resize and encode PIL image to base64 JPEG string for JSON transport."""
    img = img.copy()
    img.thumbnail((max_size, max_size), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode('utf-8')


def b64_to_pil(b64_str: str) -> Image.Image:
    """Decode base64 string to PIL image."""
    data = base64.b64decode(b64_str)
    return Image.open(io.BytesIO(data)).convert('RGB')


print('Inference utilities defined ✓')

Inference utilities defined ✓


## 9. Flask App

In [15]:
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)


@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'device': DEVICE, 'gallery_size': faiss_index.ntotal})


@app.route('/crop', methods=['POST'])
def crop_endpoint():
    """
    Input  JSON: { "image": "<base64-encoded image>" }
    Output JSON: { "crop": "<base64-encoded cropped image>", "had_detection": bool }
    """
    try:
        data     = request.get_json(force=True)
        pil_img  = b64_to_pil(data['image'])
        orig_w, orig_h = pil_img.size

        crop     = yolo_crop(pil_img)
        had_det  = (crop.size != pil_img.size)

        return jsonify({
            'crop':          pil_to_b64(crop, max_size=400),
            'had_detection': had_det,
            'original_size': [orig_w, orig_h],
            'crop_size':     list(crop.size),
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500


@app.route('/search', methods=['POST'])
def search_endpoint():
    """
    Input  JSON: { "image": "<base64 cropped image>", "top_k": int (optional) }
    Output JSON: {
        "results": [
            {
                "rank": int,
                "item_id": str,
                "cosine_score": float,
                "itm_score": float,
                "caption": str,
                "image": "<base64 gallery image>"
            }, ...
        ],
        "query_caption": str
    }
    """
    try:
        data    = request.get_json(force=True)
        top_k   = int(data.get('top_k', TOP_K_DEFAULT))
        pil_img = b64_to_pil(data['image'])

        # ── Step 1: Embed query ───────────────────────────────────────────────
        query_emb, query_caption = embed_query(pil_img, alpha=ALPHA)

        # ── Step 2: FAISS ANN retrieval — fetch ITM_TOP_N for re-ranking ─────
        n_retrieve = max(top_k, ITM_TOP_N)
        scores_faiss, indices = faiss_index.search(query_emb, n_retrieve)
        scores_faiss = scores_faiss[0]   # shape (n_retrieve,)
        indices      = indices[0]

        # ── Step 3: Gather candidate metadata & captions for ITM ─────────────
        candidates = []
        for rank_i, (idx, cos_score) in enumerate(zip(indices, scores_faiss)):
            if idx < 0 or idx >= len(gallery_data):
                continue
            g_path, g_item_id = gallery_row_to_meta[idx]
            g_caption         = caption_map.get(g_path, 'a clothing item')
            candidates.append({
                'gallery_idx':  idx,
                'item_id':      g_item_id,
                'path':         g_path,
                'cosine_score': float(cos_score),
                'caption':      g_caption,
            })

        # ── Step 4: BLIP-2 ITM re-ranking ─────────────────────────────────────
        # Score each candidate caption against the query image
        candidate_captions = [c['caption'] for c in candidates]
        itm_scores         = blip2_itm_scores(pil_img, candidate_captions)

        # Normalize both scores to [0,1] then combine: 0.6*cosine + 0.4*ITM
        cos_arr  = np.array([c['cosine_score'] for c in candidates])
        cos_norm = (cos_arr - cos_arr.min()) / (cos_arr.max() - cos_arr.min() + 1e-8)
        itm_norm = (itm_scores - itm_scores.min()) / (itm_scores.max() - itm_scores.min() + 1e-8)
        combined = 0.6 * cos_norm + 0.4 * itm_norm

        # Sort by combined score descending, take top_k
        ranked_indices = np.argsort(combined)[::-1][:top_k]

        # ── Step 5: Build response with gallery images ─────────────────────────
        results = []
        for final_rank, ci in enumerate(ranked_indices):
            c = candidates[ci]
            try:
                g_img = Image.open(c['path']).convert('RGB')
                g_img_b64 = pil_to_b64(g_img, max_size=300)
            except Exception:
                g_img_b64 = ''

            results.append({
                'rank':         final_rank + 1,
                'item_id':      c['item_id'],
                'cosine_score': round(c['cosine_score'], 4),
                'itm_score':    round(float(itm_scores[ci]), 4),
                'combined_score': round(float(combined[ci]), 4),
                'caption':      c['caption'],
                'image':        g_img_b64,
            })

        return jsonify({
            'results':       results,
            'query_caption': query_caption,
            'alpha':         ALPHA,
            'n_retrieved':   len(candidates),
        })

    except Exception as e:
        import traceback
        return jsonify({'error': str(e), 'trace': traceback.format_exc()}), 500


print('Flask app defined ✓')

Flask app defined ✓


## 10. Start Server with ngrok
**Copy the public URL printed below and paste it into your Streamlit app as `BACKEND_URL`.**

In [16]:
from pyngrok import ngrok, conf
import threading

# Set ngrok auth token
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Kill any existing tunnels
ngrok.kill()

PORT = 5000

# Open tunnel
public_url = ngrok.connect(PORT, 'http')
print('\n' + '='*60)
print(f'  BACKEND URL: {public_url}')
print('='*60)
print('Paste this URL into your Streamlit app as BACKEND_URL')
print('Health check:', f'{public_url}/health')
print()

# Run Flask in a background thread (Kaggle cells are blocking)
def run_flask():
    app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
print('Flask server running in background ✓')
print('Keep this cell (and kernel) alive while using the Streamlit app.')

                                                                                                    
  BACKEND URL: NgrokTunnel: "https://apologal-flutelike-bert.ngrok-free.dev" -> "http://localhost:5000"
Paste this URL into your Streamlit app as BACKEND_URL
Health check: NgrokTunnel: "https://apologal-flutelike-bert.ngrok-free.dev" -> "http://localhost:5000"/health

Flask server running in background ✓
Keep this cell (and kernel) alive while using the Streamlit app.
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit
127.0.0.1 - - [15/May/2026 14:58:54] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 15:03:55] "GET /health HTTP/1.1" 200 -
127.0.0.1 - - [15/May/2026 15:04:01] "POST /crop HTTP/1.1" 200 -
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Please pass a `device_map` that contains `language_model` to remove this warning. Please refer to https://github.com/huggingface/blog/blob/main/accelerate-large-models.md for more details on creating a `device_map` for large models.
127.0.0.1 - - [15/May/2026 15:04:25] "POST /search HTTP/1.1" 200 -
The `language_model` is not in the `hf_device_map` dictionary and you are running your script in a multi-GPU environment. this may lead to unexpected behavior when using `accelerate`. Ple